In [1]:
# %pip install pixell
# %pip install healpy

In [ ]:
# [5000:]

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pixell import enmap
import pandas as pd
import os

In [3]:
dir_path = '../data processing/'
output_dir = "../dataset/"
map_paths = {
    'map90': dir_path + 'act-planck_dr4dr6_coadd_AA_daynight_f090_map.fits',
    'map150': dir_path + 'act-planck_dr4dr6_coadd_AA_daynight_f150_map.fits',
    'map220': dir_path + 'act-planck_dr6.02_coadd_AA_daynight_f220_map.fits'
}

df = pd.read_csv(dir_path + "clusters_selected.csv")

os.makedirs(output_dir, exist_ok=True)

In [4]:
len(df)

9977

In [5]:
df = df[5000:]

In [6]:
len(df)

4977

In [7]:
df.head()

,name,RADeg,decDeg,redshift,redshiftType,M500c,M500cUncorr,theta500Arcmin
5000,ACT-CL J1021.6+1720,155.412352,17.340792,0.564000,b'spec',3.557584,3.935134,2.253178
5001,ACT-CL J1021.6-0138,155.413657,-1.649629,0.839000,b'phot',2.216331,2.571231,1.468210
5002,ACT-CL J1021.7+1703,155.449250,17.059663,0.570000,b'spec',1.932986,2.219395,1.829546
5003,ACT-CL J1021.9-0248,155.492337,-2.813634,0.395000,b'phot',2.324538,2.672654,2.554549
5004,ACT-CL J1022.0+0908,155.520029,9.149590,0.185504,b'spec',2.184409,2.542966,4.636959


In [8]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from pixell import enmap
from tqdm import tqdm
import os

# === Предварительные расчёты ===
df["logM500c"] = np.log10(df["M500c"])
df["logM500cUncorr"] = np.log10(df["M500cUncorr"])

diameter_arcmin = df["theta500Arcmin"].max() * 2
width_deg = diameter_arcmin / 60.0
half = width_deg / 2.0

# === Подготовим "чистые" данные для multiprocessing ===
rows = df[["RADeg", "decDeg", "redshift", "theta500Arcmin", "logM500c", "logM500cUncorr"]].to_dict(orient="records")

def process_row(row):
    try:
        ra = row["RADeg"]
        dec = row["decDeg"]
        z = row["redshift"]
        theta500 = row["theta500Arcmin"]
        massa = row["logM500c"]
        massa_uncorr = row["logM500cUncorr"]

        box = np.deg2rad([
            [dec - half, ra - half],
            [dec + half, ra + half]
        ])
        pixbox = [[0, 0], [148, 148]]

        # === Чтение карт ===
        map90 = enmap.read_map(map_paths["map90"], box=box, pixbox=pixbox)[0]
        map150 = enmap.read_map(map_paths["map150"], box=box, pixbox=pixbox)[0]
        map220 = enmap.read_map(map_paths["map220"], box=box, pixbox=pixbox)[0]

        data_image = np.stack([map90, map150, map220]).astype(np.float32)

        return data_image, z, theta500, massa, massa_uncorr

    except Exception as e:
        print(f"Ошибка при обработке (RA={row['RADeg']}, Dec={row['decDeg']}): {e}")
        return None

# === Параллельная обработка ===
images, redshifts, theta500s, logM500c, logM500cUncorr = [], [], [], [], []

max_workers = os.cpu_count()  # используй все CPU
with ProcessPoolExecutor(max_workers=4) as executor:
    for result in tqdm(executor.map(process_row, rows), total=len(rows)):
        if result is not None:
            img, z, t500, m, mu = result
            images.append(img)
            redshifts.append(z)
            theta500s.append(t500)
            logM500c.append(m)
            logM500cUncorr.append(mu)

# === Преобразуем в numpy массивы ===
images = np.array(images)
redshifts = np.array(redshifts)
theta500s = np.array(theta500s)
logM500c = np.array(logM500c)
logM500cUncorr = np.array(logM500cUncorr)

100%|██████████| 4977/4977 [1:27:44<00:00,  1.06s/it]  


In [14]:
for idx, image in enumerate(images):
    np.save(output_dir + f"images/{idx+5000}.npy", image)

In [16]:
redshifts = np.array(redshifts, dtype=np.float32) 
logM500c = np.array(logM500c, dtype=np.float32)  
logM500cUncorr = np.array(logM500cUncorr, dtype=np.float32)  

df = pd.read_csv(dir_path + "clusters_selected.csv")
redshifts = np.array(df.redshift, dtype=np.float32) 
logM500c = np.array(np.log10(df["M500c"]), dtype=np.float32)  
logM500cUncorr = np.array(np.log10(df["M500cUncorr"]), dtype=np.float32)  

np.save(output_dir + "redshifts.npy", redshifts)
np.save(output_dir + "logM500c.npy", logM500c)
np.save(output_dir + "logM500cUncorr.npy", logM500cUncorr)

In [18]:
len(redshifts)

9977

In [12]:
# for idx, image in enumerate(images):
#     print(image.shape)

(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 148, 148)
(3, 14